<a href="https://colab.research.google.com/github/shahedtheboss/breast-cancer-classification/blob/main/BreastCancerBasic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch,torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

In [ ]:
data = load_breast_cancer()
x= data.data
y= data.target
print(x.shape)
print(y.shape)
print(np.unique(y))
print(data.target_names)

(569, 30)
(569,)
[0 1]
['malignant' 'benign']


In [ ]:
xtr, xts, ytr, yts = train_test_split(
    x,y,test_size=0.2, random_state=40, stratify=y
)

In [ ]:
scalar= StandardScaler()
xtr = scalar.fit_transform(xtr)
xts = scalar.transform(xts)


In [ ]:
xtr_tensor = torch.tensor(xtr, dtype=torch.float32)
ytr_tensor = torch.tensor(ytr, dtype=torch.long)
xts_tensor = torch.tensor(xts, dtype=torch.float32)
yts_tensor = torch.tensor(yts, dtype=torch.long)

print(xtr_tensor.shape)
print(ytr_tensor.shape)
print(xts_tensor.shape)
print(yts_tensor.shape)

torch.Size([455, 30])
torch.Size([455])
torch.Size([114, 30])
torch.Size([114])


In [ ]:
class TumorNet(nn.Module):

  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(30, 8)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(8, 2)

  def forward(self,X):
    X= self.fc1(X)
    X= self.relu(X)
    X= self.fc2(X)
    return X


In [ ]:
model = TumorNet()
output=model(xtr_tensor)
print(output.shape)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.0027)


# for p in model.parameters():
#   print(name)

print(sum(p.numel() for p in model.parameters()))

torch.Size([455, 2])
266


In [ ]:
for epoch in range(20):
  model.train()
  output= model(xtr_tensor)

  loss= loss_fn(output,ytr_tensor)

  opt.zero_grad()

  loss.backward()

  opt.step()

  if (epoch+1) % 5 ==0:
    print(f"Epoch: {epoch+1} Loss: {loss.item():.4f}")


Epoch: 5 Loss: 0.5627
Epoch: 10 Loss: 0.4684
Epoch: 15 Loss: 0.3923
Epoch: 20 Loss: 0.3293


In [ ]:
model.eval()

with torch.no_grad():
  output = model(xts_tensor)

  predictions = torch.argmax(output, dim=1)
  accuracy= (predictions == yts_tensor).float().mean()
  print(f"Accuracy: {accuracy.item()}")

Accuracy: 0.9473684430122375
